# Comparing two segmentations

Segmentation is usually the pipeline choice that matters most, and it is the
hardest to check: for real tissue there are rarely reference masks to compare
against. This page works through that case — two segmentations of the same
image, no ground truth — and ends somewhere worth noticing, which is that the
metrics disagree, and the disagreement is the useful part.

## A synthetic tissue

Two lineages in spatial patches, nuclei of varying brightness imaged twice, and
lineage markers on the cell membrane. Everything is deliberate: the nuclear
restains let us ask whether a mask follows its own nucleus, the patches create
boundaries where neighbouring cells carry incompatible markers, and varying
brightness gives the per-cell quantities something to vary with.

In [1]:
import numpy as np

MARKERS = ["DNA_1", "DNA_2", "CD45", "ECAD"]
SIZE = 320


def synthetic_tissue(n_cells=120, size=SIZE, seed=0):
    """Return a (4, size, size) image, its true label mask, and the cell centres."""
    rng = np.random.default_rng(seed)
    image = rng.gamma(2.0, 8.0, size=(4, size, size)).astype(np.float32)
    truth = np.zeros((size, size), dtype=np.int32)
    yy, xx = np.ogrid[:size, :size]

    centres, placed = [], []
    while len(centres) < n_cells:
        cy, cx = rng.integers(14, size - 14, 2)
        if all((cy - py) ** 2 + (cx - px) ** 2 > 22**2 for py, px in placed):
            placed.append((cy, cx))
            centres.append((int(cy), int(cx), len(centres) + 1))

    for cy, cx, label in centres:
        nucleus = (yy - cy) ** 2 + (xx - cx) ** 2 <= 5**2
        body = (yy - cy) ** 2 + (xx - cx) ** 2 <= 9**2
        truth[body] = label
        # Nuclear content differs between cells, which is what makes agreement
        # between the two restains a meaningful quantity rather than noise.
        brightness = rng.lognormal(6.5, 0.45)
        image[0][nucleus] += brightness
        image[1][nucleus] += brightness * rng.normal(1.0, 0.04)
        # Lineage follows a spatial gradient, so like cells sit together and a
        # boundary exists where over-expansion can merge two lineages.
        immune = rng.random() < 1 / (1 + np.exp((cx - size / 2) / 25))
        image[2 if immune else 3][body & ~nucleus] += rng.lognormal(6.3, 0.35)
    return image, truth, centres


image, truth, centres = synthetic_tissue()
image.shape, truth.max()

((4, 320, 320), np.int32(120))

## Two candidate segmentations

One is conservative and stays near the nucleus; the other expands until masks
run into their neighbours. Both find every cell, so object counts cannot tell
them apart.

In [2]:
from scipy.ndimage import grey_dilation

yy, xx = np.ogrid[:SIZE, :SIZE]
tight = np.zeros_like(truth)
for cy, cx, label in centres:
    tight[(yy - cy) ** 2 + (xx - cx) ** 2 <= 7**2] = label

expanded = grey_dilation(truth, size=(13, 13))

{
    "objects (tight)": int(len(np.unique(tight)) - 1),
    "objects (expanded)": int(len(np.unique(expanded)) - 1),
    "mean area (tight)": float(np.bincount(tight.ravel())[1:].mean()),
    "mean area (expanded)": float(np.bincount(expanded.ravel())[1:].mean()),
}

{'objects (tight)': 120,
 'objects (expanded)': 120,
 'mean area (tight)': 149.0,
 'mean area (expanded)': 709.2583333333333}

Same number of cells, areas differing by a factor of three. Nothing so far says
which is better — this is precisely the position the descriptive metrics leave
you in.

## Aggregating to cell tables

Each segmentation gives a different cell-by-marker table, because the masks
decide which pixels belong to which cell.

In [3]:
import anndata as ad
import pandas as pd


def aggregate(image, labels, markers=MARKERS):
    """Mean intensity per cell per marker, as an aggregation step produces."""
    flat = labels.ravel()
    inside = flat > 0
    labs = flat[inside]
    n = int(flat.max())
    area = np.bincount(labs, minlength=n + 1)[1:].astype(float)
    present = area > 0

    X = np.zeros((int(present.sum()), len(markers)), dtype=np.float32)
    for c in range(len(markers)):
        sums = np.bincount(labs, weights=image[c].ravel()[inside], minlength=n + 1)[1:]
        X[:, c] = (sums[present] / area[present]).astype(np.float32)

    ids = np.flatnonzero(present) + 1
    obs = pd.DataFrame(
        {
            "area": area[present],
            "centroid_y": [np.argwhere(labels == i)[:, 0].mean() for i in ids],
            "centroid_x": [np.argwhere(labels == i)[:, 1].mean() for i in ids],
        },
        index=[str(i) for i in ids],
    )
    adata = ad.AnnData(X=X.copy(), obs=obs)
    adata.var_names = markers
    adata.layers["counts"] = X.copy()
    adata.layers["normalized"] = np.arcsinh(X / 5.0).astype(np.float32)
    return adata


tables = {"tight": aggregate(image, tight), "expanded": aggregate(image, expanded)}
{k: v.shape for k, v in tables.items()}

{'tight': (120, 4), 'expanded': (120, 4)}

## Scoring both

Two sources of evidence: the cell table, and the pixels. The pixel-level pass is
the expensive one, so build it once per segmentation and reuse it.

In [4]:
import sprotmetrics as spm

results = {}
for name, labels in [("tight", tight), ("expanded", expanded)]:
    table = tables[name]
    stats = spm.cell_pixel_stats(image, labels, MARKERS)
    results[name] = {
        **spm.segmentation_quality(
            table,
            nuclear_markers=["DNA_1", "DNA_2"],
            exclusive_pairs=[("CD45", "ECAD")],
        ),
        "nuclear_signal_containment": spm.nuclear_signal_containment(stats, ["DNA_1", "DNA_2"]),
        "median_signal_to_background_ratio": float(np.nanmedian(spm.signal_to_background_ratio(stats))),
    }

comparison = pd.DataFrame(results)
comparison

,tight,expanded
dapi_positive_ratio,0.375000,0.491667
replicate_correlation,0.998126,0.998079
median_replicate_cv,0.010808,0.009647
mutually_exclusive_coexpression_ratio,0.008333,0.016667
spillover_ratio,1.007551,1.064405
nuclear_signal_containment,0.848008,0.968989
median_signal_to_background_ratio,13.491301,5.939568


## Reading the result

Attach each metric's declared direction, so "higher" and "better" do not get
confused with each other.

In [5]:
catalogue = spm.list_metrics()
summary = comparison.join(catalogue[["direction", "strength"]])
summary["prefers"] = [
    "—"
    if not np.isfinite(row.tight) or not np.isfinite(row.expanded) or row.tight == row.expanded
    else ("tight" if (row.tight > row.expanded) == (row.direction == "higher is better") else "expanded")
    for row in summary.itertuples()
]
# How far apart are they? A direction with no magnitude behind it is not evidence.
summary["rel_diff"] = (summary.tight - summary.expanded).abs() / summary[["tight", "expanded"]].abs().max(axis=1)
summary[["tight", "expanded", "rel_diff", "direction", "strength", "prefers"]].round(4)

,tight,expanded,rel_diff,direction,strength,prefers
dapi_positive_ratio,0.3750,0.4917,0.2373,higher is better,weak,expanded
replicate_correlation,0.9981,0.9981,0.0000,higher is better,weak,tight
median_replicate_cv,0.0108,0.0096,0.1074,lower is better,strong,expanded
mutually_exclusive_coexpression_ratio,0.0083,0.0167,0.5000,lower is better,strong,tight
spillover_ratio,1.0076,1.0644,0.0534,lower is better,strong,tight
nuclear_signal_containment,0.8480,0.9690,0.1249,higher is better,weak,expanded
median_signal_to_background_ratio,13.4913,5.9396,0.5597,higher is better,strong,tight


This is the part worth slowing down for. The metrics **do not agree**, and
sorting out why is the whole job.

Three prefer the tight segmentation, and they are measuring the same underlying
failure from different angles:

- `spillover_ratio` — intensity bleeding across cell boundaries.
- `mutually_exclusive_coexpression_ratio` — single objects positive for two
  markers that no one cell should carry at once, which is what a mask spanning
  two cells produces.
- `median_signal_to_background_ratio` — expanding a mask adds pixels that are
  mostly background, diluting the signal inside it.

Two prefer the **expanded** one, and they are not wrong so much as answering a
different question. `dapi_positive_ratio` and `nuclear_signal_containment` both
rise whenever masks grow, because a larger mask is likelier to contain nuclear
signal — including a neighbour's. A single mask covering the entire image would
score 1.0 on both. They detect debris objects well and over-expansion not at
all, so neither can carry this decision alone.

And one, `median_replicate_cv`, nominally prefers `expanded` — look at its
`rel_diff` before believing it. A direction with no magnitude behind it is not
evidence, and the `prefers` column above will happily report a winner for a
difference in the fourth decimal place.

The package marks the gameable ones rather than leaving them to be
rediscovered:

In [6]:
info = spm.get_metric("nuclear_signal_containment")
print(f"direction: {info.direction.value}")
print(f"strength : {info.strength.value}")
print()
print(info.notes)

direction: higher is better
strength : weak

Must be read against spillover_ratio, never alone. Growing every mask raises it, so on exemplar-001 it prefers watershed_expand (0.67) to threshold (0.44) while spillover_ratio says watershed over-expands (1.21 against 1.09). The pair brackets segmentation; either one by itself is gameable.


`Strength.WEAK` marks metrics that are directional but cannot carry a decision
on their own. Filtering to the strong directional evidence, and requiring the
difference to be large enough to mean something:

In [7]:
strong = summary[
    (summary.strength == "strong")
    & summary.direction.isin(["higher is better", "lower is better"])
    & (summary.rel_diff > 0.05)
]
strong[["tight", "expanded", "rel_diff", "prefers"]].round(4)

,tight,expanded,rel_diff,prefers
median_replicate_cv,0.0108,0.0096,0.1074,expanded
mutually_exclusive_coexpression_ratio,0.0083,0.0167,0.5000,tight
spillover_ratio,1.0076,1.0644,0.0534,tight
median_signal_to_background_ratio,13.4913,5.9396,0.5597,tight


Three of the four point to `tight`, and they are the ones measuring mask
boundaries: signal crossing them, lineages merging across them, background
diluting what is inside them. That matches how the data was built — `expanded`
dilates masks well past the cells they belong to — so the metrics recovered the
right answer without ever seeing the ground truth.

The dissenter deserves a second look rather than a shrug. `median_replicate_cv`
reports 0.0108 against 0.0096, which the relative filter waves through at 11%
— but both numbers are a hair above zero. They say the two nuclear restains
agree within a cell to about one percent in *both* segmentations, which is true
and unsurprising: both masks contain the nucleus, so both capture the same
restain signal. A relative difference between two near-zero values exaggerates;
the absolute difference, 0.001, is the honest figure. The metric is not
disagreeing so much as declining to discriminate.

That is a general hazard, not a quirk of this example. Ratio filters mislead
near zero, and the fix is to look at the raw values rather than to tune the
threshold until the answer comes out tidy.

Note what this did *not* require: a single composite score. Three agreeing
measurements of boundary violation are a stronger argument than any weighted
average of all seven, which would have let two size-driven metrics pull against
the three that actually detect the failure.

## If you do have reference masks

Then you can measure agreement directly, and should. Here `truth` is available
because we generated it.

In [8]:
pd.DataFrame(
    {name: spm.segmentation_agreement(labels, truth) for name, labels in [("tight", tight), ("expanded", expanded)]}
).loc[
    [
        "foreground_precision",
        "foreground_recall",
        "foreground_dice",
        "object_f1_at_iou_10",
        "object_f1_at_iou_50",
        "merge_rate",
    ]
]

,tight,expanded
foreground_precision,1.000000,0.356711
foreground_recall,0.588933,1.000000
foreground_dice,0.741294,0.525846
object_f1_at_iou_10,1.000000,1.000000
object_f1_at_iou_50,1.000000,0.041667
merge_rate,0.000000,0.075000


Precision and recall separate the two failure modes: the expanded segmentation
keeps high recall — it is not missing cells — while precision collapses, because
most of what it claims is not cell. That is over-expansion, stated directly.

Note how the two agree. The reference-free metrics preferred `tight`, and so
does Dice against the truth. That is the argument for trusting them on real
tissue, where this last check is not available.